# MMLU-TR: benchmark flow

Builds the benchmark step by step, using the code in `funcs.py`. The earlier trials are in `example.ipynb`.

The method follows Stanford HELM's MMLU leaderboard, with the prompt translated into Turkish. Each model receives the prompt as plain text and generates one token at temperature 0. The answer is correct when that token is the right letter.

## 1. Dataset

The dataset version is pinned to a Hub revision in `data_versions/mmlu_tr_questions_v1.json`, so the same version always loads the same data. `load_data` downloads it, using the local cache after the first time, and checks each split's row count.

- `dev`: the 5 example questions per subject
- `test`: the questions models are evaluated on
- `validation`: not used

In [1]:
from tr_ai_evals.benchmarks.mmlu_tr.funcs import load_data

DATASET_NAME = "mmlu_tr_questions"
DATASET_VERSION = "v1"

data, dataset_revision = load_data(name=DATASET_NAME, version=DATASET_VERSION)

print(f"Revision: {dataset_revision}")
for split, rows in data.items():
    print(f"{split}: {len(rows):,} questions, {len({row['subject'] for row in rows})} subjects")

Revision: d2c8c04923d7b5a62608ffc58d8e582bc3666e08
dev: 285 questions, 57 subjects
validation: 1,531 questions, 57 subjects
test: 14,042 questions, 57 subjects


In [2]:
import pandas as pd

test = pd.DataFrame(data["test"])
print(f"Fields: {list(test.columns)}")
test.head()

Fields: ['id', 'subject', 'question', 'choices', 'answer', 'question_en', 'choices_en']


,id,subject,question,choices,answer,question_en,choices_en
0,abstract_algebra_test_0001,abstract_algebra,"Q(sqrt(2), sqrt(3), sqrt(18))'nin Q üzerindeki...","[0, 4, 2, 6]",B,Find the degree for the given field extension ...,"[0, 4, 2, 6]"
1,abstract_algebra_test_0002,abstract_algebra,"S_5 içinde p = (1, 2, 5, 4)(2, 3) olsun. S_5 i...","[8, 2, 24, 120]",C,"Let p = (1, 2, 5, 4)(2, 3) in S_5 . Find the i...","[8, 2, 24, 120]"
2,abstract_algebra_test_0003,abstract_algebra,Katsayıları bu sonlu cisimde olan verilen poli...,"[0, 1, 0,1, 0,4]",D,Find all zeros in the indicated finite field o...,"[0, 1, 0,1, 0,4]"
3,abstract_algebra_test_0004,abstract_algebra,İfade 1 | Abel olmayan bir grubun bölüm grubu ...,"[Doğru, Doğru, Yanlış, Yanlış, Doğru, Yanlış, ...",B,Statement 1 | A factor group of a non-Abelian ...,"[True, True, False, False, True, False, False,..."
4,abstract_algebra_test_0005,abstract_algebra,Verilen polinom halkasında verilen polinomları...,"[2x^2 + 5, 6x^2 + 4x + 6, 0, x^2 + 1]",B,Find the product of the given polynomials in t...,"[2x^2 + 5, 6x^2 + 4x + 6, 0, x^2 + 1]"


## 2. Example questions

`select_shots` chooses and orders each subject's 5 example questions from `dev` exactly as HELM does, so every model sees the same examples.

In [3]:
from tr_ai_evals.benchmarks.mmlu_tr.funcs import select_shots

shots = select_shots(data["dev"])

print(f"{len(shots)} subjects, {set(len(rows) for rows in shots.values())} examples each")
print("anatomy:", [(row["id"], row["answer"]) for row in shots["anatomy"]])

57 subjects, {5} examples each
anatomy: [('anatomy_dev_0001', 'D'), ('anatomy_dev_0002', 'D'), ('anatomy_dev_0005', 'B'), ('anatomy_dev_0003', 'C'), ('anatomy_dev_0004', 'C')]


## 3. Prompt

The prompt template and the Turkish subject names are versioned files:

- `templates/mmlu_tr_template_v1.json`: the header and the `Soru` and `Cevap` labels
- `subjects/mmlu_tr_subjects_v1.json`: each subject's Turkish name

Each loader returns the content and its 8-character hash, which goes into the manifest.

In [4]:
from tr_ai_evals.benchmarks.mmlu_tr.funcs import build_prompt, load_versioned

TEMPLATE_NAME = "mmlu_tr_template"
TEMPLATE_VERSION = "v1"
SUBJECTS_NAME = "mmlu_tr_subjects"
SUBJECTS_VERSION = "v1"

template, template_hash = load_versioned("templates", TEMPLATE_NAME, TEMPLATE_VERSION)
subject_names, subjects_hash = load_versioned("subjects", SUBJECTS_NAME, SUBJECTS_VERSION)
assert set(subject_names) == set(shots), "Every subject needs a Turkish name."

item = next(row for row in data["test"] if row["subject"] == "anatomy")
print(build_prompt(template, subject_names[item["subject"]], shots[item["subject"]], item))

Aşağıdakiler anatomi hakkında çoktan seçmeli sorulardır (cevaplarıyla birlikte).

Soru: Hiyoid kemiğinin embriyolojik kökeni nedir?
A. Birinci faringeal ark
B. Birinci ve ikinci faringeal arklar
C. İkinci faringeal ark
D. İkinci ve üçüncü faringeal arklar
Cevap: D

Soru: Trigeminal sinirin bu dallarından hangileri somatik motor lifler içerir?
A. Supraorbital sinir
B. İnfraorbital sinir
C. Mental sinir
D. Yukarıdakilerin hiçbiri
Cevap: D

Soru: Aşağıdakilerden hangisi hipofiz bezini içeren vücut boşluğudur?
A. Karın boşluğu
B. Kafatası boşluğu
C. Plevra boşluğu
D. Omurilik boşluğu
Cevap: B

Soru: Plevralarla ilgili aşağıdakilerden hangisi doğrudur?
A. Duyusal innervasyonları yoktur.
B. Birbirlerinden 2 mm'lik bir aralıkla ayrılırlar.
C. Boyuna uzanırlar.
D. Solunum epitelinden oluşurlar.
Cevap: C

Soru: Angle Sınıf II Bölüm 2 oklüzyonunda aşağıdakilerden hangisi görülür?
A. Üst yan kesici dişlerde aşırı dikey örtüşme.
B. Üst orta kesici dişlerde negatif yatay örtüşme.
C. Üst yan kesici 

## 4. Generation settings

`params/mmlu_tr_params_v1.json` holds the Ollama options: temperature 0, one generated token, and a context length that fits the longest prompt.

In [5]:
PARAMS_NAME = "mmlu_tr_params"
PARAMS_VERSION = "v1"

params, params_hash = load_versioned("params", PARAMS_NAME, PARAMS_VERSION)
print(params)

{'temperature': 0, 'num_predict': 1, 'num_ctx': 8192}


## 5. Model

The digest identifies the model's exact weights.

In [6]:
from tr_ai_evals.benchmarks.mmlu_tr.funcs import get_model_digest

MODEL = "gemma3:4b"

model_digest = get_model_digest(MODEL)
print(MODEL, model_digest)

gemma3:4b a2af6cc3eb7fa8be8504abaf9b04e88f17a119ec3f04a3addf55f92841195f5a


## 6. Manifest

The manifest records everything that defines a run, and is saved next to its answers.

In [7]:
import json

from tr_ai_evals.benchmarks.mmlu_tr.funcs import build_manifest

# Experiment 0 is for trials in this notebook; real experiments start at 1.
EXPERIMENT = 0

manifest = build_manifest(
    experiment=EXPERIMENT,
    model=MODEL,
    model_digest=model_digest,
    dataset=(DATASET_NAME, DATASET_VERSION, dataset_revision),
    template=(TEMPLATE_NAME, TEMPLATE_VERSION, template_hash),
    subjects=(SUBJECTS_NAME, SUBJECTS_VERSION, subjects_hash),
    params=(PARAMS_NAME, PARAMS_VERSION, params_hash),
    shots=shots,
)
print(json.dumps(manifest, ensure_ascii=False, indent=2))

{
  "timestamp": "2026-11-17T07:36:35.517308+00:00",
  "experiment": 0,
  "method": "helm_plain_text",
  "model": "gemma3:4b",
  "model_digest": "a2af6cc3eb7fa8be8504abaf9b04e88f17a119ec3f04a3addf55f92841195f5a",
  "dataset": {
    "name": "mmlu_tr_questions",
    "version": "v1",
    "revision": "d2c8c04923d7b5a62608ffc58d8e582bc3666e08"
  },
  "template": {
    "name": "mmlu_tr_template",
    "version": "v1",
    "hash": "54f2d89c"
  },
  "subjects": {
    "name": "mmlu_tr_subjects",
    "version": "v1",
    "hash": "ec352b6a"
  },
  "params": {
    "name": "mmlu_tr_params",
    "version": "v1",
    "hash": "e2bc6187"
  },
  "shots": {
    "per_subject": 5,
    "hash": "ecf93939"
  },
  "git": {
    "commit": "03e7f87",
    "dirty": true
  }
}


## 7. Asking the model

`ask_model` sends the prompt as plain text, without the model's chat template, using the generation settings. It returns the generated text and the prompt's length in tokens. `parse_answer` reads the letter; anything other than A, B, C or D is an invalid answer.

In [8]:
import ollama

from tr_ai_evals.benchmarks.mmlu_tr.funcs import ask_model, parse_answer


def make_prompt(question: dict) -> str:
    return build_prompt(template, subject_names[question["subject"]], shots[question["subject"]], question)


reply = ask_model(ollama.Client(), MODEL, make_prompt(item), params)
print(item["id"], "correct answer:", item["answer"])
print(reply, "->", parse_answer(reply["response"]))

anatomy_test_0001 correct answer: A
{'response': ' C', 'prompt_tokens': 508} -> C


## 8. Running and saving

`run_experiment` asks every question and saves each answer as it arrives, to `experiments/experiment_<number>/<run id>/`:

- `manifest.json`: the manifest from step 6, plus the run id
- `answers.jsonl`: one line per question, with the raw generated text, the parsed letter, the correct answer, the prompt length, any error and the time taken

The run id is the model name plus a hash of the settings that define the run, so the same settings always continue the same run. Stopping and running again continues where it stopped.

A failed request, such as Ollama being down, counts as wrong and isn't asked again unless `retry_failed=True`. The run stops if a prompt was probably cut at the context limit, after 50 failures in a row, or if accuracy is below 10% after 100 questions.

This trial runs 2 questions per subject. Running it again shows the run continuing with nothing left to ask.

In [9]:
from tr_ai_evals.benchmarks.mmlu_tr.funcs import run_experiment

QUESTIONS_PER_SUBJECT = 2

sample = []
for subject in sorted(shots):
    sample += [row for row in data["test"] if row["subject"] == subject][:QUESTIONS_PER_SUBJECT]

summary = run_experiment(manifest, sample, make_prompt, params)

Experiment 0, run gemma3-4b_dad4f0d7: 0 done, 114 not asked, 0 failed.


Finished: {'run_id': 'gemma3-4b_dad4f0d7', 'total': 114, 'asked': 114, 'failed': 0, 'invalid': 0, 'not_asked': 0, 'correct': 62, 'accuracy': 0.543859649122807}


In [10]:
summary = run_experiment(manifest, sample, make_prompt, params)

Experiment 0, run gemma3-4b_dad4f0d7: 114 done, 0 not asked, 0 failed.
Finished: {'run_id': 'gemma3-4b_dad4f0d7', 'total': 114, 'asked': 114, 'failed': 0, 'invalid': 0, 'not_asked': 0, 'correct': 62, 'accuracy': 0.543859649122807}


## 9. Reading results

`load_experiment` returns every run in an experiment with the latest answer for each question.

In [11]:
from tr_ai_evals.benchmarks.mmlu_tr.funcs import load_experiment, summarize_answers

for run_id, (run_manifest, records) in load_experiment(EXPERIMENT).items():
    print(run_id, summarize_answers({record["question_id"]: record for record in records}, len(sample)))

answers = pd.DataFrame(next(iter(load_experiment(EXPERIMENT).values()))[1])
answers.head()

gemma3-4b_dad4f0d7 {'total': 114, 'asked': 114, 'failed': 0, 'invalid': 0, 'not_asked': 0, 'correct': 62, 'accuracy': 0.543859649122807}


,question_id,subject,response,prediction,correct_answer,prompt_tokens,error,seconds
0,abstract_algebra_test_0001,abstract_algebra,B,B,B,458,None,0.076
1,abstract_algebra_test_0002,abstract_algebra,A,A,C,469,None,0.037
2,anatomy_test_0001,anatomy,C,C,A,508,None,0.075
3,anatomy_test_0002,anatomy,B,B,B,498,None,0.038
4,astronomy_test_0001,astronomy,A,A,A,780,None,0.075
